# Clase 17 — Introducción a contenedores y Docker

**Módulo:** De local a producción: contenedores, nube y operación.

**Objetivo de la lectura y discusión:** explicar cómo Docker empaca y ejecuta una aplicación, distinguir imágenes y contenedores, e interpretar el Dockerfile, las rutas, la caché y el propósito de Compose.

## Dinámica de la clase: lectura, discusión y explicación

### 1. Lectura individual completa

Lee **todo el contenido teórico del notebook**, no sólo la sección que después corresponda a tu equipo. Incluye las explicaciones, tablas y ejemplos de Dockerfile, contexto y rutas, imágenes de Python, comandos, puertos, caché y Docker Compose. Hay explicaciones teóricas tanto antes como después de la práctica: continúa hasta la recopilación final.

Revisa los ejemplos de comandos para interpretar qué hacen. **La ejecución de la práctica se realizará en la siguiente sesión**; durante esta lectura no necesitas construir imágenes ni ejecutar contenedores.

Mientras lees, anota una idea importante que puedas explicar con tus propias palabras, una duda que quieras contrastar y una predicción sobre qué ocurriría al construir, ejecutar o modificar un contenedor.

### 2. Formación de equipos y asignación de secciones

El grupo de 14 estudiantes se organizará en **cinco equipos: cuatro de tres integrantes y uno de dos**. **El profesor formará los equipos y les comunicará durante la clase la sección que corresponde a cada uno**, después de la lectura individual.

Las cinco secciones para preparar son:

| Sección | Enfoque de la explicación |
|---|---|
| Imagen y contenedor | Qué problema resuelven, cómo se relacionan y qué agregan respecto a `uv.lock`. |
| Dockerfile | Cómo interpretar el ejemplo, elegir la imagen base y distinguir construcción de ejecución. |
| Contexto y rutas | Cómo se relacionan el contexto, `COPY`, `WORKDIR` y la ubicación del Dockerfile. |
| Caché y reconstrucción | Qué se reutiliza cuando cambia el código y por qué una imagen nueva no actualiza un contenedor activo. |
| Docker Compose | Qué aporta para una aplicación con API e interfaz y cómo se diferencia del Dockerfile. |

### 3. Preparación y discusión en equipo

Comparen sus interpretaciones y dudas sobre la lectura completa. Después, **cada equipo debe preparar la sección que le fue asignada**: una explicación con sus propias palabras, un ejemplo concreto del notebook y una pregunta para discutir con todo el grupo. Distribuyan la participación entre sus integrantes.

**Son libres de preparar material adicional si lo requieren**, como diagramas, diapositivas, ejemplos o referencias que ayuden a explicar su sección. No es obligatorio elaborar diapositivas. Si utilizan recursos externos, indiquen su fuente.

### 4. Explicación y discusión con el grupo

Cada equipo explicará su sección y planteará su pregunta. Los demás participarán con respuestas razonadas, preguntas, ejemplos o argumentos que complementen o cuestionen lo presentado. Contrasten las explicaciones con el notebook y justifiquen sus conclusiones.

Al finalizar, revisa tus notas: ¿qué idea puedes explicar ahora con mayor claridad y qué duda sigue pendiente? El profesor cerrará la discusión y aclarará los conceptos que lo necesiten.

## Punto de partida

Antes de trabajar, abre la terminal en la raíz del repositorio público, revisa `git status`, confirma que estás en `main` y ejecuta `git pull`. Si aparecen cambios sobre material rastreado, revísalos antes de actualizar; no los descartes. Después ejecuta `uv sync --locked`.

**Prerrequisitos:** Python 3.12, `uv`, HTTP y JSON, [FastAPI y Streamlit de Clase 14](../modulo-02-ciclo-mlops/clase-14-model-service-streamlit.ipynb), configuración personal de Databricks y acceso al modelo de Clase 13. Docker se verifica en la sección de instalación.

**Resultado de la lectura y discusión:** explicar los conceptos con ejemplos y justificar las predicciones sobre construcción, ejecución y reconstrucción.

**Resultado de la práctica de la siguiente sesión:** una imagen de la API, un contenedor accesible desde el host, una predicción válida y una segunda versión que permite observar reconstrucción y caché. La práctica es local y no requiere commit, push, PR ni entrega en Canvas.

## Aprendiendo a utilizar Docker

Una API puede funcionar en tu ambiente y fallar en otra computadora: faltan bibliotecas, cambia Python o el comando se ejecuta desde otra carpeta. `uv.lock` fija las dependencias de Python; un contenedor agrega un entorno de ejecución con archivos y bibliotecas del sistema.

Docker permite construir ese entorno y ejecutar la aplicación como un proceso aislado. Empacar no elimina las dependencias externas: esta API continúa necesitando red y autorización para cargar el modelo registrado.

### Video: What is Docker in 5 Minutes

TechSquidTV explica visualmente el problema que resuelven los contenedores.

[Ver el video en YouTube](https://www.youtube.com/watch?v=_dfLOzuIg2o&t=17s).

## ¿Qué es Docker?

La API necesita Python, bibliotecas y código. Streamlit necesita su propio proceso para atender la interfaz. ¿Cómo ejecutar cada componente con su entorno y conservarlo al cambiar de computadora?

Docker permite construir, distribuir y ejecutar esos entornos mediante imágenes y contenedores.

### Contenedor: la aplicación en ejecución

Un **contenedor** es una instancia aislada en la que se ejecuta una aplicación con sus archivos y dependencias. Por ejemplo, la API puede ejecutarse en un contenedor y Streamlit en otro. Cada uno tiene su proceso principal y puede iniciarse o detenerse por separado.

El **host** es la computadora o entorno que ejecuta Docker. El contenedor tiene su propio espacio de archivos y configuración: no ve automáticamente todas las carpetas, variables o credenciales del host.

Un ambiente virtual de Python separa paquetes; un contenedor incluye además los archivos y bibliotecas del sistema necesarios para la aplicación. A diferencia de una máquina virtual, no tiene un kernel propio: comparte el kernel del entorno que lo ejecuta. Docker Desktop proporciona ese entorno Linux en Windows y macOS.

Puedes detener un contenedor y conservarlo; detenerlo no es lo mismo que eliminarlo. Los archivos escritos sólo dentro de ese contenedor se pierden al eliminarlo.

Referencia: [¿Qué es un contenedor? — Docker](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/).

### Imagen: el paquete desde el que se crea el contenedor

Una **imagen** contiene los archivos y la configuración necesarios para crear contenedores: en nuestra API, Python, las bibliotecas instaladas y `api.py`. Descargar una imagen no inicia la aplicación; ejecutarla crea un contenedor.

Dos propiedades ayudan a entender su funcionamiento:

- **Es inmutable:** para incorporar cambios construyes otra imagen. Editar `api.py` en el host no cambia el código que ya está dentro de una imagen.
- **Está formada por capas:** la construcción puede añadir archivos y dependencias por pasos. Docker puede reutilizar resultados de pasos anteriores cuando sus entradas no cambian.

Desde una misma imagen puedes crear varios contenedores con configuración y estado propios. El contenido de la imagen no incluye un kernel independiente; para ejecutarla necesitas un entorno compatible con su sistema y arquitectura.

Referencia: [¿Qué es una imagen? — Docker](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-an-image/).

### Dockerfile: las instrucciones de construcción

Un **Dockerfile** es un archivo de texto, sin extensión, que describe cómo construir una imagen: qué base utilizar, qué archivos copiar, cómo instalar dependencias y qué comando ejecutar al arrancar el contenedor.

```text
Dockerfile ── construir ──> Imagen ── ejecutar ──> Contenedor
```

El Dockerfile es la descripción; la imagen es el resultado empacado; el contenedor es una instancia creada desde ese resultado.

Referencia: [escribir un Dockerfile — Docker](https://docs.docker.com/get-started/docker-concepts/building-images/writing-a-dockerfile/).

### Registro de imágenes y Docker Hub

Un **registro** almacena y distribuye imágenes. Docker Hub es uno de esos registros y aloja, entre otras, la imagen oficial de Python. Descargar una imagen se llama *pull* y subirla a un registro se llama *push*.

El **Model Registry de MLflow** gestiona versiones de modelos; el registro de Docker distribuye imágenes del entorno que ejecuta la aplicación. La imagen de nuestra API y el modelo registrado son dos artefactos distintos.

Referencia: [¿Qué es un registro? — Docker](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-registry/).

### Archivos copiados y datos persistentes

Los archivos que copies al construir forman parte de la imagen. Si la aplicación necesita guardar datos que sobrevivan a la eliminación del contenedor, puede utilizar un **volumen**, almacenamiento administrado por Docker. Un **bind mount** conecta una carpeta del host con una ruta del contenedor.

La API de esta práctica utiliza código copiado en la imagen y recibe su configuración al arrancar. El acceso al modelo remoto sigue dependiendo de la red y de tus permisos.

## 🧠 Analogía general: la pastelería de Docker

| Concepto | Analogía | Explicación técnica |
|---|---|---|
| **Dockerfile** | La receta con ingredientes y pasos | Archivo de texto con instrucciones para crear una imagen. |
| **Imagen** | El pastel congelado | Artefacto empacado con archivos, Python y dependencias. |
| **Contenedor** | Una porción servida del pastel | Instancia de una imagen con un proceso y estado propios. |
| **Docker** | La cocina industrial | Herramientas que construyen imágenes y ejecutan contenedores. |

El Dockerfile dice cómo preparar; la imagen es el resultado preparado; el contenedor ejecuta ese resultado. Una misma imagen permite crear varios contenedores.

**Límite de la analogía:** cada contenedor tiene su propia capa de escritura; los cambios de uno no modifican la imagen ni aparecen automáticamente en los demás.

### Otra perspectiva: el molde y las galletas

La analogía de [Dean Pleban, *Docker for Data Science — A Step by Step Guide* (2020)](https://medium.com/data-science/docker-for-data-science-a-step-by-step-guide-1e5f7f3baf8e) permite distinguir la plantilla de las instancias que se crean a partir de ella:

| Concepto | Analogía del molde | Relación en Docker |
|---|---|---|
| **Dockerfile** | Instrucciones para fabricar el molde | Describe cómo construir la imagen. |
| **Imagen** | Molde reutilizable para cortar galletas | Sirve como base para crear varios contenedores. |
| **Contenedor** | Cada galleta producida con el molde | Es una instancia con configuración y estado propios. |

En esta comparación, el Dockerfile representa las instrucciones para fabricar el **molde**, no la receta para hornear las galletas. Construir una imagen y crear un contenedor son operaciones diferentes.

Si modificas las instrucciones, debes construir una nueva imagen; los contenedores existentes no cambian automáticamente. Al cambiar el código de la API, necesitas reconstruir la imagen y crear un contenedor desde ella para ejecutar la nueva versión.

**Hasta dónde llega la comparación:** una galleta es un objeto estático; un contenedor ejecuta procesos cuando está en marcha. Además, compartir una imagen significa partir de los mismos archivos y valores predeterminados, no tener siempre el mismo comportamiento o estado: cada contenedor puede recibir distinta configuración y datos, y acumular cambios en su propia capa de escritura. La imagen original permanece intacta. Consulta también la explicación oficial de Docker sobre [imágenes e inmutabilidad](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-an-image/).

### Componentes de Docker

Una imagen, un contenedor y el motor de Docker cumplen funciones diferentes. La infografía reúne estos componentes junto con el Dockerfile y los volúmenes.

<img src="../assets/modulo-03-produccion/clase-16/componentes-docker-infografia.png" alt="Componentes de Docker: imágenes, contenedores, Dockerfile, motor, daemon y volúmenes" width="640">

*Infografía: Nikki Siapno y Chris Staudinger, Level Up Coding. [Publicación de la autora](https://www.linkedin.com/posts/nikkisiapno_%F0%9D%97%A7%F0%9D%97%B5%F0%9D%97%B2-%F0%9D%97%BA%F0%9D%97%AE%F0%9D%97%B6%F0%9D%97%BB-%F0%9D%97%B0%F0%9D%97%BC%F0%9D%97%BA%F0%9D%97%BD%F0%9D%97%BC%F0%9D%97%BB%F0%9D%97%B2%F0%9D%97%BB%F0%9D%98%81%F0%9D%98%80-%F0%9D%97%BC%F0%9D%97%B3-activity-7168195532383793152-1KqS).*

El **daemon** es el servicio que permanece en segundo plano y recibe las solicitudes del cliente. El comando `docker` es ese cliente; el motor administra las imágenes y los contenedores.

![El cliente envía solicitudes al motor; el motor obtiene imágenes del registro y crea contenedores](../assets/modulo-03-produccion/clase-16/componentes-docker.svg)

*El cliente solicita operaciones; el motor las ejecuta y obtiene las imágenes del registro cuando las necesita.*

La imagen es el paquete de archivos y configuración; el contenedor es la instancia creada desde ella. Tener el cliente instalado no demuestra que el motor esté disponible.

<a id="ejemplo-dockerfile"></a>

## Ejemplo de Dockerfile

Este Dockerfile construye el entorno de una API que vive en `api.py`. Parte de una imagen con Python, instala dependencias, copia el código y define cómo arrancar el servidor. `requirements.txt` contiene dependencias generadas desde el lock del proyecto.

```dockerfile
FROM python:3.12-slim

WORKDIR /code

COPY requirements.txt /code/requirements.txt
RUN python -m pip install --no-cache-dir -r requirements.txt

COPY . /code/

EXPOSE 8000
CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]

```

| Instrucción | Qué significa |
|---|---|
| `FROM` | Elige la imagen base; aquí Python 3.12 sobre Linux. |
| `WORKDIR` | Establece el directorio para instrucciones posteriores y el proceso de inicio. Lo crea si hace falta. |
| `COPY` | Copia archivos desde el contexto de construcción a la imagen. |
| `RUN` | Ejecuta un comando durante la construcción. |
| `EXPOSE` | Documenta el puerto esperado de la aplicación; no lo publica en el host. |
| `CMD` | Define el comando por defecto al crear y arrancar un contenedor. La lista separa programa y argumentos. |

`python -m pip` ejecuta el instalador con ese intérprete; `-r` lee requisitos y `--no-cache-dir` evita guardar la caché de descargas de pip dentro de la imagen. **Esta opción de pip no desactiva el caché de construcción de Docker.**

`uvicorn` arranca el servidor ya trabajado con FastAPI. `api:app` significa: importar `api.py` y usar su objeto `app`. `--host` define dónde escucha y `--port` el puerto interno. El servidor arranca al ejecutar el contenedor, no durante `docker build`.

<a id="contexto-y-rutas"></a>

## Antes de construir: contexto, Dockerfile y rutas

En el [Dockerfile anterior](#ejemplo-dockerfile), `WORKDIR /code` y las dos instrucciones `COPY` conectan los archivos de tu computadora con los de la imagen. Para interpretar esas rutas necesitas responder dos preguntas: **¿dónde está el archivo en tu computadora y dónde debe quedar dentro de la imagen?** El contexto responde la primera; el destino de `COPY` y `WORKDIR` responden la segunda.

### 1. La carpeta de la terminal y el contexto

La **carpeta actual del host** es donde estás al escribir el comando; `pwd` permite consultarla. El **contexto de construcción** es el conjunto de archivos disponibles para construir la imagen, después de aplicar las exclusiones de `.dockerignore`. Este archivo contiene patrones de archivos y carpetas que no deben enviarse al constructor.

Una construcción de imagen también se llama **build**. En un build con archivos locales, el argumento final selecciona ese contexto:

```text
docker build [opciones] CONTEXTO
```

`[opciones]` y `CONTEXTO` son marcadores de la sintaxis. Estos ejemplos seleccionan una carpeta concreta:

| Desde dónde escribes el comando | Argumento de contexto | Carpeta que Docker puede utilizar |
|---|---|---|
| `labs/trabajo-local/clase-16/` | `.` | La carpeta actual, `clase-16/`. |
| Raíz del repositorio | `labs/trabajo-local/clase-16` | La misma carpeta `clase-16/`. |

La opción `-t nyc-taxi-api:v1` asigna un nombre y una etiqueta a la imagen: `nyc-taxi-api` es el nombre y `v1` es la etiqueta o *tag*. No selecciona una carpeta.

**El punto `.` también define explícitamente el contexto.** Escribir sólo `docker build -t nyc-taxi-api:v1` deja fuera el argumento necesario y produce un error; Docker no deduce el contexto a partir del tag ni de `WORKDIR`.

### 2. ¿Dónde busca Docker el Dockerfile?

Si no utilizas `-f`, Docker busca un archivo llamado `Dockerfile` en la raíz del contexto. La opción `-f` —también llamada `--file`— permite seleccionar otro archivo; su ruta local se interpreta desde la carpeta actual de la terminal.

Por ejemplo, desde la raíz del repositorio:

```bash
docker build -t nyc-taxi-api:v1 labs/trabajo-local/clase-16
```

El contexto es `labs/trabajo-local/clase-16` y Docker busca allí `Dockerfile`. El mismo caso, indicando también el archivo, se escribe:

```bash
docker build -f labs/trabajo-local/clase-16/Dockerfile -t nyc-taxi-api:v1 labs/trabajo-local/clase-16
```

**`-f` selecciona las instrucciones; el último argumento selecciona los archivos disponibles.** Pueden apuntar a ubicaciones diferentes. En estos dos comandos apuntan a la misma carpeta de trabajo.

### 3. `COPY`: del contexto a la imagen

La forma básica es `COPY origen destino`. En este ejemplo, `api.py` debe existir en la raíz del contexto:

```dockerfile
COPY api.py /code/api.py
```

- `api.py` es una ruta relativa **al contexto**, no a la carpeta del Dockerfile ni a `WORKDIR`.
- `/code/api.py` es una ruta absoluta **dentro de la imagen Linux**, incluso si construyes desde Windows.
- Si el archivo queda fuera del contexto o está excluido por `.dockerignore`, no está disponible para esa copia. Cambiar `WORKDIR` no lo hace aparecer.

Si seleccionaras la raíz del repositorio como contexto, ese mismo `COPY api.py ...` buscaría `api.py` en la raíz del repositorio. Indicar el Dockerfile con `-f labs/trabajo-local/clase-16/Dockerfile` no cambia esa búsqueda. Por eso conviene que el contexto sea la carpeta pequeña que contiene los archivos de la API.

### 4. `WORKDIR`: el directorio dentro de la imagen

`WORKDIR /code` establece `/code` como directorio de trabajo para las instrucciones posteriores y para el proceso al arrancar. Si no existe, lo crea. No mueve la terminal del host ni cambia las rutas de origen de `COPY`.

```dockerfile
WORKDIR /code
COPY api.py .
```

Aquí el destino `.` es relativo a `WORKDIR`: el archivo queda en `/code/api.py`. Compara estas variantes cuando `WORKDIR` es `/code`:

| Copia | Origen en el host | Destino dentro de la imagen |
|---|---|---|
| `COPY api.py /code/api.py` | `api.py` en el contexto | `/code/api.py` |
| `COPY api.py .` | `api.py` en el contexto | `/code/api.py` |
| `COPY . .` | Contenido permitido del contexto | Contenido dentro de `/code/` |

En `COPY . .`, cada símbolo `.` pertenece a un espacio diferente: **el primero es el contexto y el segundo es el directorio de trabajo de la imagen**. Una ruta relativa de `WORKDIR` también depende del directorio anterior; utilizar `/code` como ruta absoluta evita esa ambigüedad.

### 5. Relacionar las rutas con el arranque

```text
Host: labs/trabajo-local/clase-16/       Imagen:
├── Dockerfile                         /code/
├── requirements.txt                   ├── requirements.txt
├── api.py                             └── api.py
├── .dockerignore
└── .env                               (excluido de la copia)
```

Con `WORKDIR /code`, `uvicorn api:app` puede importar `/code/api.py` y utilizar su objeto `app`. `RUN` ejecuta un comando durante la construcción. Si ese comando lee `requirements.txt` mediante una ruta relativa, lo busca desde `/code`, dentro de la imagen que se está construyendo.

En el siguiente ejemplo, `PurePosixPath` representa las rutas sin acceder al disco. Observa cómo el mismo nombre de archivo se resuelve en el contexto del host y en el directorio de la imagen.

Referencias: [contexto de construcción](https://docs.docker.com/build/concepts/context/), [`COPY`](https://docs.docker.com/reference/dockerfile/#copy), [`WORKDIR`](https://docs.docker.com/reference/dockerfile/#workdir) y [opción `-f`](https://docs.docker.com/reference/cli/docker/buildx/build/#file).

In [ ]:
from pathlib import PurePosixPath

contexto_host = PurePosixPath("labs/trabajo-local/clase-16")
directorio_imagen = PurePosixPath("/code")

origen = contexto_host / "api.py"
destino = directorio_imagen / "api.py"
print("Origen de COPY:", origen)
print("Destino de COPY:", destino)
assert str(destino) == "/code/api.py"

## Elegir una imagen base de Python

La primera línea del Dockerfile, `FROM python:3.12-slim`, selecciona una imagen oficial que ya contiene Python. Para elegirla conviene distinguir **la versión de Python**, **la variante de la imagen** y **su sistema base**.

### 1. Leer el nombre y la etiqueta

En `python:3.12-slim`, los dos puntos separan el repositorio de la **etiqueta** o *tag*:

| Parte | Significado |
|---|---|
| `python` | Repositorio de la imagen oficial. |
| `3.12` | Versión de Python seleccionada. |
| `slim` | Variante reducida de la imagen. |

La etiqueta completa es `3.12-slim`. La práctica conserva Python **3.12** tanto en el host como en el contenedor.

### 2. Comparar las variantes Linux

| Variante | Qué incluye o cambia | Qué revisar al elegirla |
|---|---|---|
| Estándar: `python:3.12` | Más herramientas y bibliotecas de sistema preinstaladas. | Útil si necesitas compilar dependencias; ocupa más espacio. |
| Reducida: `python:3.12-slim` | Menos paquetes de sistema sobre una base Debian. | Puede necesitar bibliotecas adicionales; es la base de esta API. |
| Alpine: `python:3.12-alpine` | Otra distribución Linux, con la biblioteca de sistema musl en lugar de glibc. | Algunas dependencias científicas necesitan paquetes distintos o compilación. |

**Elección para la API:** `slim` permite partir de una base reducida y comprobar las dependencias con el mismo ejemplo del curso. Reducir la imagen base no reduce automáticamente las dependencias de la aplicación: aquí instalamos las exportadas del lock del curso.

### 3. Distinguir distribución y sistema operativo

Una etiqueta también puede precisar la distribución, por ejemplo `python:3.12-slim-bookworm`: mantiene Python 3.12 y la variante `slim`, y especifica Debian Bookworm. Esto permite elegir la base del sistema con mayor precisión.

Las variantes Windows Server Core pertenecen a otra familia de contenedores y requieren compatibilidad específica con Windows. La práctica utiliza **contenedores Linux**, también cuando Docker Desktop se ejecuta en Windows o macOS.

### 4. No confundir etiqueta con contenido inmutable

Un tag puede apuntar a una imagen actualizada. El identificador de imagen permite distinguir el contenido local. Tampoco existe un tamaño fijo por variante: depende de la arquitectura, la versión y los paquetes añadidos. Elegir la imagen exige comprobar compatibilidad además de tamaño.

Referencia: [imagen oficial de Python y sus variantes](https://hub.docker.com/_/python).


## ¿Qué es Docker Compose?

Una aplicación puede necesitar varios componentes: **Streamlit recibe los datos y la API produce la predicción**. Si cada componente se ejecuta en un contenedor, hay que conservar su imagen, puertos y configuración, y permitir la comunicación entre ellos.

**Docker Compose** es una herramienta para definir y ejecutar los servicios de una aplicación mediante un archivo `compose.yaml`. Un **servicio** describe cómo ejecutar un componente, por ejemplo la API o la interfaz. YAML es un formato de texto que organiza propiedades mediante claves y sangría.

```text
Navegador → Streamlit → HTTP → API → Modelo
            servicio UI       servicio API
```

### Dockerfile y Compose cumplen funciones distintas

| Archivo | Pregunta que responde | Ejemplo |
|---|---|---|
| `Dockerfile` | ¿Cómo se construye una imagen? | Instalar dependencias y copiar `api.py`. |
| `compose.yaml` | ¿Cómo se ejecutan y conectan los servicios? | Elegir imágenes, puertos, variables y conexiones para API y UI. |

Un servicio puede utilizar una imagen existente o indicar un Dockerfile y un contexto para construirla. Compose también puede utilizarse con un solo servicio.

### Leer una configuración pequeña

Este fragmento muestra únicamente la construcción de la API; no es todavía la configuración completa para ejecutar API y Streamlit:

```yaml
services:
  api:
    build:
      context: ./labs/trabajo-local/clase-16
      dockerfile: Dockerfile
```

Si `compose.yaml` estuviera en la raíz del repositorio:

- `services` agrupa los componentes y `api` es el nombre elegido para este servicio.
- `build` indica cómo construir su imagen; `context` selecciona los archivos disponibles para la construcción.
- `dockerfile` selecciona el archivo de instrucciones **dentro de ese contexto**. En este ejemplo puede omitirse porque su nombre es `Dockerfile`.

La sangría expresa qué propiedades pertenecen a cada nivel; no son comandos de Python. La construcción utiliza los mismos archivos de la API descritos en el Dockerfile.

### Ejecutar y conectar los servicios

Con una configuración completa, `docker compose up` construye las imágenes que haga falta construir, crea o inicia los contenedores y prepara su red. `docker compose down` detiene y retira los contenedores y la red creados para esa aplicación. La API y Streamlit siguen siendo procesos separados que se comunican por HTTP.

En la red compartida de Compose, los servicios pueden encontrarse por su nombre. Por ejemplo, Streamlit podría llamar a `http://api:8000` si el servicio de API se llama `api` y escucha en el puerto interno 8000. Desde el contenedor de Streamlit, `localhost` apunta al propio contenedor de la interfaz.

**Idea clave:** el Dockerfile describe la construcción de cada imagen; Compose reúne la configuración necesaria para ejecutar los componentes como una aplicación.

Referencias: [¿Qué es Docker Compose?](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-docker-compose/), [configuración de construcción](https://docs.docker.com/reference/compose-file/build/) y [aplicaciones con varios contenedores](https://docs.docker.com/get-started/docker-concepts/running-containers/multi-container-applications/).


## Instalación de Docker Desktop o Docker Engine

Los comandos de la práctica se escriben en **Git Bash en Windows, Zsh en macOS o Bash en Linux**, no en una celda Python. No necesitan activar un ambiente para ejecutar el cliente Docker.

### Windows

1. Abre la [guía oficial de Docker Desktop para Windows](https://docs.docker.com/desktop/setup/install/windows-install/), verifica requisitos de tu versión y descarga el instalador compatible con tu arquitectura.
2. Sigue el asistente y utiliza el backend WSL 2 cuando corresponda. Si solicita habilitar virtualización, actualizar WSL o reiniciar, completa esas acciones antes de continuar. Los comandos administrativos de Windows se ejecutan en la terminal indicada por esa guía, no se sustituyen por comandos de Bash.
3. Abre Docker Desktop y espera a que el motor esté listo. Usa **contenedores Linux**.
4. Reabre Git Bash si el programa no se encuentra en `PATH`.

### macOS

1. Abre la [guía oficial para Mac](https://docs.docker.com/desktop/setup/install/mac-install/) y elige el instalador para **Intel** o **Apple silicon**, según tu equipo.
2. Abre el instalador, mueve Docker a Aplicaciones y abre Docker Desktop.
3. Completa las autorizaciones del asistente y espera a que el motor esté listo; reabre la terminal si no encuentra el cliente.

### Linux

Utiliza la [guía de instalación de Docker Engine](https://docs.docker.com/engine/install/) para tu distribución; [Ubuntu](https://docs.docker.com/engine/install/ubuntu/) y [Debian](https://docs.docker.com/engine/install/debian/) tienen instrucciones distintas. Sigue sus pasos para repositorio de paquetes, instalación y arranque del servicio. Si tu instalación de Engine requiere permisos administrativos, antepone `sudo` a los comandos `docker` de esta práctica: por ejemplo, `sudo docker info` o `sudo docker ps`. `sudo` ejecuta el comando con privilegios administrativos; `uv` se ejecuta con tu usuario habitual. No cambies los permisos del socket para evitar esta comprobación.

### Verificación rápida

```bash
docker --version
docker info
```

`--version` muestra la versión del cliente; `info` debe consultar al motor y mostrar sus datos. La verificación está completa si el cliente responde y `docker info` termina sin un error de conexión o permisos.

Si aparece `command not found`, revisa instalación y `PATH`; si dice `Cannot connect`, abre Docker Desktop o revisa el servicio de Engine; si falla la descarga, revisa la red. Si el error persiste, conserva el mensaje completo para identificar si corresponde a instalación, permisos o red.

## Primeros comandos útiles

| Comando | Qué permite observar o hacer |
|---|---|
| `docker images` | Lista imágenes locales, tags, identificadores y tamaño. |
| `docker ps` | Lista contenedores activos. |
| `docker ps -a` | Incluye contenedores detenidos; `-a` significa todos. |
| `docker logs nombre` | Consulta la salida del proceso de un contenedor existente. |
| `docker stop nombre` | Solicita detener el proceso del contenedor indicado. |
| `docker rm nombre` | Elimina un contenedor detenido. |
| `docker rmi imagen:tag` | Quita la referencia de imagen indicada; puede requerir retirar sus contenedores primero. |

`nombre` e `imagen:tag` son marcadores de explicación: se sustituyen por recursos concretos que identifiques con `ps` e `images`. Antes de eliminar, confirma que pertenecen a esta práctica.

Para consultar ayuda propia del programa utiliza `docker run --help` o `docker build --help`.

## Construir la imagen de Docker

Una construcción o **build** lee el Dockerfile y produce una imagen. `-t` asigna nombre y etiqueta; el argumento final indica la carpeta del contexto.

```bash
docker build -t nyc-taxi-api:v1 .
```

Este ejemplo se ejecuta dentro de la carpeta que contiene Dockerfile, requisitos y API. Si falla `COPY`, verifica tanto el archivo como la carpeta que pasaste como contexto. Docker no puede copiar cualquier archivo de tu computadora.

## Ejecutar el contenedor

Crear la imagen no inicia la API. `docker run` crea un contenedor nuevo y ejecuta su comando `CMD`. La opción `--env-file` permite cargar las variables de `.env` al arrancar el contenedor, sin incluir ese archivo en la imagen.

Estas opciones controlan cómo se ejecuta el contenedor:

| Opción | Efecto |
|---|---|
| `--name` | Asigna un nombre al contenedor, independiente del nombre de imagen. |
| `--env-file .env` | Lee variables desde ese archivo del host. No las convierte en un mecanismo para compartir secretos. |
| `-p 127.0.0.1:8000:8000` | Publica en el host local el puerto interno 8000. |
| `-d` | Deja el proceso ejecutándose en segundo plano; la terminal devuelve el control. |
| `--rm` | Elimina el contenedor al terminar; después ya no puedes consultar sus logs con `docker logs`. |

Para depurar un arranque fallido puede convenir omitir temporalmente `--rm`, consultar estado y logs, y eliminar luego ese contenedor concreto.

### Puertos, `localhost` y dirección de escucha

El formato completo de publicación es `DIRECCION_HOST:PUERTO_HOST:PUERTO_CONTENEDOR`.

- Uvicorn escucha en `0.0.0.0` **dentro del contenedor** para aceptar peticiones por su interfaz de red.
- El host publica sólo en `127.0.0.1` para acceder desde tu propia computadora.
- El navegador utiliza `http://127.0.0.1:8000`; `0.0.0.0` no es la dirección de destino que debes copiar.
- `EXPOSE 8000` documenta; `-p` publica. Son acciones distintas.

`localhost` es «este entorno». Desde el host apunta al host; desde otro contenedor apunta a ese otro contenedor. Si se mueve Streamlit a un contenedor, su dirección de API debe revisarse.

![Una solicitud del host atraviesa el puerto publicado y llega a Uvicorn dentro del contenedor](../assets/modulo-03-produccion/clase-16/puertos-api.svg)

*El puerto del host puede cambiar sin cambiar el puerto donde escucha Uvicorn.*

## Primer contenedor: una web lista para ejecutar

Antes de construir la API, ejecuta una imagen que ya contiene una aplicación web. El ejemplo de la [guía oficial de Docker](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/) permite observar la diferencia entre descargar una imagen, ejecutar un contenedor y acceder a su servicio.

En tu terminal:

```bash
docker run -d --rm --name docker-bienvenida -p 127.0.0.1:8080:80 docker/welcome-to-docker
docker ps
```

Si la imagen no está en tu computadora, Docker la descarga. `run` crea el contenedor y `-d` lo deja en segundo plano. La aplicación escucha en el puerto **80 dentro del contenedor**; `-p` la hace accesible en el **8080 de tu host**. El nombre `docker-bienvenida` permite identificar esta instancia.

Abre `http://127.0.0.1:8080` en el navegador. Debe aparecer la página de bienvenida de Docker. Si el puerto 8080 está ocupado, puedes publicar `127.0.0.1:8081:80` y abrir 8081; el puerto interno sigue siendo 80.

Observa las dos entidades:

```bash
docker images docker/welcome-to-docker
docker logs docker-bienvenida
```

La primera lista muestra la **imagen almacenada**. Los logs pertenecen al **contenedor en ejecución**. Para terminar:

```bash
docker stop docker-bienvenida
docker ps -a
```

Como utilizaste `--rm`, el contenedor `docker-bienvenida` ya no debe aparecer, pero la imagen sigue disponible para crear otro.

**Checkpoint:** abriste una web servida por un contenedor, distinguiste su imagen y explicaste los dos puertos. La API NYC Taxi utilizará el mismo mecanismo.

## 🛠️ Práctica: estructura del proyecto y contenedor de la API

### 1. Preparar la carpeta local

Desde la **raíz del repositorio público**, prepara el espacio ignorado por Git:

```bash
mkdir -p labs/trabajo-local/clase-16
cp labs/starters/clase-16-docker/api.py labs/trabajo-local/clase-16/api.py
```

`mkdir -p` crea la ruta si falta; `cp` copia la API funcional. Si ya trabajaste en esa carpeta, revisa sus archivos antes de repetir la copia porque `cp` puede reemplazar `api.py`.

```text
labs/trabajo-local/clase-16/
├── api.py
├── requirements.txt    (se genera desde el lock)
├── Dockerfile          (se crea en la práctica)
├── .dockerignore       (se crea en la práctica)
└── .env                (configuración personal, no se versiona)
```

El código base conserva [la API de Clase 14](../modulo-02-ciclo-mlops/clase-14-model-service-streamlit.ipynb): cinco features, Pipeline completo, `/health` y `/api/v1/predicciones`. Consulta el archivo: `VERSION_API` empieza en `v1` y aparece en `/health`; no modifica el modelo. La carga remota resuelve el alias una vez y carga esa versión concreta, para que los metadatos coincidan con el modelo aunque se mueva el alias después.

`load_dotenv(override=False)` permite leer un `.env` local sin reemplazar variables que ya recibió el proceso; dentro de Docker la configuración entra mediante `--env-file`.

**Checkpoint:** la copia existe en la carpeta local; el starter original permanece sin editar.

### 2. Generar `requirements.txt` desde el lock de `uv`

`requirements.txt` es un formato que puede leer pip dentro de la imagen. **No es una segunda fuente manual de dependencias:** se genera desde `uv.lock`.

Desde la raíz del repositorio:

```bash
uv export --locked --no-emit-project --no-hashes --output-file labs/trabajo-local/clase-16/requirements.txt
```

`export` convierte el lock a otro formato; no instala bibliotecas. `--locked` exige que el lock esté actualizado sin modificarlo; `--no-emit-project` exporta dependencias sin pedir instalar este repositorio como paquete; `--no-hashes` omite hashes de archivos para simplificar la lectura; `--output-file` indica la salida.

La exportación incluye todas las dependencias del ambiente del curso, también herramientas como Jupyter o Streamlit que esta API no utiliza. Esto aumenta el tamaño de la imagen. Conserva el archivo generado sin editarlo a mano: `pyproject.toml` y `uv.lock` siguen definiendo las dependencias del proyecto.

Abre el archivo generado: contiene versiones fijadas y posibles condiciones por plataforma. No copies `.venv` a la imagen; el ambiente del host puede pertenecer a otro sistema operativo.

### 3. Preparar la configuración externa

Crea `.env` en `labs/trabajo-local/clase-16/` con VS Code. Usa los nombres de [`.env.example`](../labs/starters/clase-16-docker/.env.example) y recupera **tu propia configuración que ya funciona con Clase 14**. No uses los valores de otra persona.

| Variable | Valor que debes revisar |
|---|---|
| `MLFLOW_TRACKING_URI` | `databricks` |
| `MLFLOW_REGISTRY_URI` | `databricks-uc` |
| `MODELO_REGISTRADO` | Modelo de Clase 13 al que ya tienes acceso. |
| `ALIAS_MODELO` | `champion`, si existe para ese modelo. |
| `DATABRICKS_HOST` | URL de tu workspace. |
| `DATABRICKS_TOKEN` | Tu credencial vigente, sólo en el archivo local. |

Escribe cada variable como `NOMBRE=valor`, sin `export` ni comillas envolventes. No guardes credenciales en el Dockerfile, código, capturas o imagen. La ruta remota y la autorización siguen siendo necesarias: Docker no hereda tu sesión ni lee automáticamente archivos del host.

**Alternativa con un modelo local:** si recibes del profesor un MLflow Model local completo, coloca su carpeta en `artefactos/modelo/` dentro del trabajo local y configura `RUTA_MODELO_LOCAL=artefactos/modelo` en `.env`. La API usa ese artefacto en lugar del Registry; reporta `local` como versión y `sin-alias`. En esta modalidad, las predicciones se calculan con el artefacto local y no requieren conexión al Registry.

### 4. Crear `.dockerignore` y el Dockerfile

Crea `.dockerignore` dentro de `labs/trabajo-local/clase-16/` con este contenido:

```text
.env
.env.*
.venv/
.git/
__pycache__/
*.pyc
*.ipynb

```

Cada patrón excluye archivos del contexto enviado al constructor. `.gitignore` no sustituye a `.dockerignore`. `*` representa una coincidencia de nombres; `.env.*` evita incluir variantes del archivo de configuración. `__pycache__` y `*.pyc` son cachés locales de Python.

Crea después `Dockerfile`, sin `.txt`, con el [ejemplo explicado arriba](#ejemplo-dockerfile). Conserva el orden requisitos → instalación → código para aprovechar el caché.

El comando principal será `uvicorn api:app`: la copia final debe dejar `api.py` en `/code/`. La construcción no solicita predicciones ni descarga el modelo del Registry; eso ocurre al arrancar la API.

**Checkpoint:** Dockerfile y `.dockerignore` están en la misma carpeta que `api.py` y `requirements.txt`. El archivo `.env` queda excluido.

### 5. Construir desde dos ubicaciones

Desde la raíz del repositorio, entra en la carpeta de trabajo y verifica su contenido:

```bash
cd labs/trabajo-local/clase-16
pwd
ls -a
```

`pwd` muestra la carpeta actual y `ls -a` incluye los archivos que empiezan con punto. Debes encontrar `Dockerfile`, `api.py`, `requirements.txt`, `.dockerignore` y `.env`.

#### A. Usar la carpeta actual como contexto

```bash
docker build -t nyc-taxi-api:v1 .
docker images nyc-taxi-api
```

El punto selecciona `clase-16/` como contexto. Como no hay `-f`, Docker lee el `Dockerfile` de esa carpeta. Su `COPY requirements.txt ...` busca el archivo en esa raíz del contexto.

La primera descarga e instalación pueden tardar. El resultado esperado es una imagen con repositorio `nyc-taxi-api` y tag `v1`.

#### B. Seleccionar archivo y contexto desde la raíz del repositorio

Sin modificar ningún archivo, regresa tres niveles: de `clase-16/` a `trabajo-local/`, de ahí a `labs/` y después a la raíz del repositorio.

```bash
cd ../../..
pwd
ls
```

Comprueba que ves `pyproject.toml`, `uv.lock`, `labs/` y los módulos del curso. Construye indicando el archivo y el contexto:

```bash
docker build -f labs/trabajo-local/clase-16/Dockerfile -t nyc-taxi-api:v1 labs/trabajo-local/clase-16
```

| Parte del comando | Función |
|---|---|
| `-f labs/trabajo-local/clase-16/Dockerfile` | Selecciona las instrucciones desde la ubicación de la terminal. |
| `-t nyc-taxi-api:v1` | Asigna el mismo nombre y tag a la imagen resultante. |
| `labs/trabajo-local/clase-16` al final | Selecciona el mismo contexto que antes. |

Los dos builds utilizan los mismos archivos y Dockerfile. Si no cambió ninguna entrada y el caché sigue disponible, verás pasos `CACHED` en la segunda construcción.

**Pregunta de diagnóstico:** si conservaras `-f` pero cambiaras el último argumento por `.`, ¿dónde buscaría `COPY requirements.txt ...`? En la raíz del repositorio. El archivo generado está en `labs/trabajo-local/clase-16/`, por lo que esa copia fallaría. Elegir el Dockerfile no selecciona automáticamente su carpeta como contexto.

Regresa a la carpeta donde están `.env` y los archivos de la API:

```bash
cd labs/trabajo-local/clase-16
pwd
```

**Checkpoint:** puedes señalar la carpeta de la terminal, el archivo Dockerfile, el contexto y el destino `/code/` dentro de la imagen. Los siguientes comandos se ejecutan desde esta carpeta de trabajo.

Una imagen construida no es todavía una API en ejecución. El tag `v1` tampoco es la versión del modelo de MLflow.

### 6. Ejecutar el contenedor

Comprueba antes que ninguna API anterior esté usando el puerto 8000. Si hay una, detén su proceso desde su propia terminal o elige otro puerto del host.

```bash
docker run -d --rm --name nyc-taxi-api-v1 --env-file .env -p 127.0.0.1:8000:8000 nyc-taxi-api:v1
docker ps
docker logs nyc-taxi-api-v1
```

`-d` devuelve el control de la terminal; `--name` identifica esta instancia y `--rm` la elimina al terminar. Repite `logs` si la carga del modelo sigue en curso. Un identificador devuelto por `run` no demuestra que el arranque haya terminado: espera el mensaje de aplicación iniciada.

Si la API termina antes de consultar los logs, repite el comando **sin `-d`**, conserva `--rm` y observa el error directamente. Si falla un nombre ya ocupado, inspecciona `docker ps -a` antes de decidir qué detener o eliminar.

**Checkpoint:** el contenedor sigue activo, los logs muestran el arranque y el navegador abre `http://127.0.0.1:8000/health`. Debe responder `status: ok` y `version_api: v1`.

### 7. Comprobar una predicción

Abre `http://127.0.0.1:8000/docs`. Expande `POST /api/v1/predicciones`, pulsa **Try it out**, reemplaza el body por este ejemplo y pulsa **Execute**:

```json
{
  "distancia_km": 4.2,
  "pasajeros": 2,
  "hora_recoleccion": 18,
  "zona_origen": 75,
  "zona_destino": 42
}
```

También puedes usar Postman con método POST, la misma URL y un body JSON. La distancia se expresa en kilómetros, como exige el contrato de la API.

**Resultado esperado:** HTTP `200` con `duracion_estimada_minutos`, `modelo`, `alias` y `version`. El número depende del modelo cargado. El campo `version` identifica el modelo de MLflow. El tag `v1` identifica la imagen y `version_api` en `/health` identifica la versión del código de la API.

Repite cambiando `pasajeros` a `0`: debe responder `422`. Pydantic continúa validando el mismo contrato dentro del contenedor.

Si el puerto 8000 estaba ocupado, al ejecutar utiliza `-p 127.0.0.1:8001:8000` y abre el navegador en 8001; Uvicorn sigue escuchando en 8000 dentro del contenedor. No cambies todas las referencias al mismo tiempo.

## Construcción incremental y cacheo en Docker

### ¿Cómo funciona el caché?

La construcción procesa instrucciones en orden y puede reutilizar resultados previos. `RUN` y `COPY` pueden añadir capas de archivos; otras instrucciones, como `CMD`, definen metadatos. No todas las instrucciones crean una capa de archivos.

Para `COPY`, Docker examina los archivos incluidos; para un `RUN` ordinario, no vuelve a consultar Internet sólo porque pasó tiempo. Si cambia una entrada necesaria para un paso, se invalida ese paso y los posteriores que dependen de él en esta construcción lineal.

### Impacto de la posición de los comandos

Este orden reinstala dependencias cuando cambia cualquiera de los archivos copiados:

```dockerfile
COPY . /code/
RUN python -m pip install --no-cache-dir -r requirements.txt
```

En cambio, el Dockerfile de la práctica copia requisitos, instala dependencias y después copia la API. Si sólo cambia `api.py`, puede reutilizar la instalación. El caché ahorra trabajo; no convierte una imagen en código editable ni actualiza contenedores en ejecución.

![Al cambiar sólo el código se reutiliza la instalación de dependencias; al cambiar los requisitos se repite la instalación y la copia del código](../assets/modulo-03-produccion/clase-16/cache-construccion.svg)

*Con la misma imagen base y caché disponible, el orden de las instrucciones determina qué resultados se reutilizan.*

### Comprobar una segunda versión

En tu copia local de `api.py`, cambia sólo `VERSION_API = "v1"` por `VERSION_API = "v2"`. Construye desde la misma carpeta:

```bash
docker build -t nyc-taxi-api:v2 .
```

Busca `CACHED` en los pasos reutilizados. El paso de copiar código debe reflejar el cambio. Antes de detener nada, vuelve a `/health`: el contenedor existente debe seguir mostrando **v1**.

```bash
docker stop nyc-taxi-api-v1
docker run -d --rm --name nyc-taxi-api-v2 --env-file .env -p 127.0.0.1:8000:8000 nyc-taxi-api:v2
```

Espera el arranque y vuelve a comprobar `/health` y una predicción. Ahora debe mostrar **v2**. Se conserva el contrato y el modelo seleccionado por configuración; si alguien mueve el alias antes del nuevo arranque, también puede cambiar la versión del modelo, por lo que debes leer sus metadatos.

**Checkpoint:** identificas qué se reconstruyó y por qué fue necesario crear otro contenedor.

### Construcción incremental y `RUN`

Dos comandos relacionados pueden escribirse en un `RUN` usando `&&`: en el shell Linux de la construcción, ejecuta el segundo sólo si el primero termina correctamente.

```dockerfile
RUN comando_a && comando_b
```

Los nombres anteriores son marcadores, no instrucciones para copiar al Dockerfile. Combinar operaciones puede evitar pasos intermedios innecesarios, pero no corrige un orden equivocado ni reduce el tamaño por sí solo. 

### Etiquetas y limpieza explícita

```bash
docker images nyc-taxi-api
docker ps -a
```

Comprueba qué imagen y contenedor pertenecen al ejercicio. Usar otra vez `:v1` puede mover ese tag a una nueva imagen; la imagen anterior puede quedar sin etiqueta. **No hay borrado automático garantizado por reutilizar un tag.**

Cuando termines y ya hayas leído los logs:

```bash
docker stop nyc-taxi-api-v2
docker rmi nyc-taxi-api:v1 nyc-taxi-api:v2
```

`--rm` elimina el contenedor detenido; `rmi` retira estas referencias de imagen. Las capas compartidas y el caché pueden seguir ocupando espacio. No ejecutes limpiezas globales para resolver un recurso concreto sin inspeccionarlo. Si necesitas conservar la práctica para revisarla, puedes mantener las imágenes hasta terminar esa revisión.

## Reto opcional: mismo servicio, otro puerto

Ejecuta una imagen ya construida en otro puerto libre del host, conservando el puerto interno de la API. Decide un nombre de contenedor distinto y comprueba `/health` y una predicción. Explica qué parte de la publicación cambió y cuál no.

Consulta [publicación de puertos](https://docs.docker.com/get-started/docker-concepts/running-containers/publishing-ports/) y [`docker run`](https://docs.docker.com/reference/cli/docker/container/run/). Detén y elimina sólo el recurso adicional que creaste. Si ya retiraste la imagen, reconstrúyela primero.

## Errores frecuentes

| Síntoma | Qué revisar primero |
|---|---|
| `docker` no se encuentra | Instalación, terminal reabierta y `PATH`. |
| No conecta al motor | Docker Desktop abierto o servicio de Engine disponible. |
| `COPY` no encuentra un archivo | Último argumento del build, ubicación relativa al contexto y patrones de `.dockerignore`; `-f` sólo selecciona el Dockerfile. |
| No importa `api` | `/code/api.py`, `WORKDIR` y `api:app`. |
| Credencial o modelo no disponible | `.env` local, permisos, red y nombre/alias del Registry; no el tag de imagen. |
| Puerto ocupado | Proceso o contenedor anterior en el host; cambiar sólo el puerto publicado si corresponde. |
| El navegador no conecta | Estado del contenedor, logs, arranque completo, escucha en `0.0.0.0` y publicación. |
| Contenedor desapareció | Con `--rm` se elimina al terminar; ejecutar en primer plano para leer el error. |
| Sigue apareciendo v1 | Se construyó v2, pero el contenedor activo sigue usando v1. |
| Poco espacio en disco | Tamaño de imágenes, contenedores detenidos y caché; inspeccionar antes de borrar. |

## Recopilación

El Dockerfile describe la construcción; la imagen contiene el entorno empacado; el contenedor ejecuta un proceso con configuración y estado propios. El lock conserva la selección de dependencias Python y `.dockerignore` controla los archivos enviados al build.

Una petición atraviesa el puerto del host y llega al puerto interno de Uvicorn. El modelo conserva su contrato de cinco features. Reconstruir genera una imagen; para observar el cambio hay que crear un contenedor desde ella. Ordenar dependencias antes del código permite reutilizar caché sin confundirlo con actualización automática.

La carpeta del host, el contexto, el Dockerfile elegido y `WORKDIR` cumplen funciones distintas. `COPY` lee desde el contexto; su destino y las rutas de ejecución pertenecen a la imagen. Compose permite describir la ejecución de varios servicios sin confundir esa descripción con la construcción de cada imagen.

## ✅ Check final de la práctica

- [ ] Distingo Dockerfile, imagen, contenedor y registry de imágenes frente a Model Registry.
- [ ] Identifico carpeta del host, Dockerfile, contexto, directorio interno y módulo de arranque.
- [ ] Construí desde la carpeta local y desde la raíz indicando el mismo contexto.
- [ ] Construí una imagen con tag y sin copiar `.env` ni `.venv`.
- [ ] Comprobé salud, predicción válida y rechazo `422`.
- [ ] Expliqué el puerto del host, el interno y el significado de `localhost`.
- [ ] Observé caché, reconstruí y comprobé v1 → v2 al recrear el contenedor.
- [ ] Consulté logs antes de cerrar y sé retirar sólo mis recursos.
- [ ] Distingo el propósito de Dockerfile y `compose.yaml`.

## Referencias

- Docker: [conceptos de contenedores](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/), [Dockerfile](https://docs.docker.com/reference/dockerfile/) y [contexto y `.dockerignore`](https://docs.docker.com/build/concepts/context/).
- Docker: [publicación de puertos](https://docs.docker.com/get-started/docker-concepts/running-containers/publishing-ports/), [caché](https://docs.docker.com/build/cache/invalidation/), [tags](https://docs.docker.com/reference/cli/docker/image/tag/) y [limpieza](https://docs.docker.com/engine/manage-resources/pruning/).
- Dean Pleban (2020): [*Docker for Data Science — A Step by Step Guide*](https://medium.com/data-science/docker-for-data-science-a-step-by-step-guide-1e5f7f3baf8e), fuente de la analogía del molde y las galletas.
- Docker: [imágenes e inmutabilidad](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-an-image/), [registros](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-registry/), [escribir un Dockerfile](https://docs.docker.com/get-started/docker-concepts/building-images/writing-a-dockerfile/) y [Docker Compose](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-docker-compose/).
- Docker: [`COPY`](https://docs.docker.com/reference/dockerfile/#copy), [`WORKDIR`](https://docs.docker.com/reference/dockerfile/#workdir) y [seleccionar el Dockerfile con `-f`](https://docs.docker.com/reference/cli/docker/buildx/build/#file).
- Nikki Siapno y Chris Staudinger, Level Up Coding: [componentes de Docker](https://www.linkedin.com/posts/nikkisiapno_%F0%9D%97%A7%F0%9D%97%B5%F0%9D%97%B2-%F0%9D%97%BA%F0%9D%97%AE%F0%9D%97%B6%F0%9D%97%BB-%F0%9D%97%B0%F0%9D%97%BC%F0%9D%97%BA%F0%9D%97%BD%F0%9D%97%BC%F0%9D%97%BB%F0%9D%97%B2%F0%9D%97%BB%F0%9D%98%81%F0%9D%98%80-%F0%9D%97%BC%F0%9D%97%B3-activity-7168195532383793152-1KqS).
- TechSquidTV: [*What is Docker in 5 Minutes*](https://www.youtube.com/watch?v=_dfLOzuIg2o&t=17s).
- Astral: [exportar el lockfile](https://docs.astral.sh/uv/concepts/projects/export/).
- [Imagen oficial de Python](https://hub.docker.com/_/python).

Código y explicaciones propios bajo MIT. Las referencias externas conservan sus licencias.